In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate


c:\code\Documentreader\ragenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load pdf file


### PyPDFLoader was chosen because:

1. Simple API - just pass file path and call .load()
2. Extracts selectable text directly from PDFs
3. Preserves metadata (page numbers, source file info)
4. Well-integrated with LangChain ecosystem
5. Good for text-based PDFs (not scanned images)

##### Alternative PDF loaders in LangChain:

- PDFPlumberLoader: Better text extraction, handles tables/layouts
- PDFMinerLoader: More robust parsing, slower
- UnstructuredPDFLoader: Handles complex layouts, images, tables
- AmazonTextractPDFLoader: AWS-based, handles scanned PDFs with OCR
- LlamaParseLoader: AI-powered extraction, excellent for complex PDFs

##### Trade-offs:

PyPDFLoader: Fast, simple, lightweight - best for clean text PDFs
Others: More features but slower, more dependencies, higher costs (some)

For this project, PyPDFLoader is ideal because:

- Academic paper is text-based and clean
- Performance is good for RAG workflows
- Minimal setup needed for proof-of-concept

print("PDF Loader comparison documented. Using PyPDFLoader for this academic paper.")


In [2]:
#loading data
loader = PyPDFLoader("data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf")
document = loader.load()
print(document)

Ignoring wrong pointing object 18 0 (offset 0)


[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creator': 'Preview', 'creationdate': "D:20240909152042Z00'00'", 'author': 'Thu Vu', 'moddate': "D:20240910141854Z00'00'", 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology', 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='APPLIED COGNITIVE PSYCHOLOGY\nAppl. Cognit. Psychol. 20: 139–156 (2006)\nPublished online 31 October 2005 in Wiley InterScience\n(www.interscience.wiley.com) DOI: 10.1002/acp.1178\nConsequences of Erudite Vernacular Utilized Irrespective\nof Necessity: Problems with Using Long Words Needlessly\nDANIEL M. OPPENHEIMER*\nPrinceton University, USA\nSUMMARY\nMost texts on writing style encourage authors to avoid overly-complex words. However, a majority\nof undergraduates admit to deliberately increasing the complexity of their vocabulary so as to give\nthe impression of intelligen

# Split text


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,length_function=len,
                                            separators=["\n\n", "\n", " "])
doc = text_splitter.split_documents(document)
print(len(doc))
print(doc)
print(doc[1].page_content)

24
[Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creator': 'Preview', 'creationdate': "D:20240909152042Z00'00'", 'author': 'Thu Vu', 'moddate': "D:20240910141854Z00'00'", 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology', 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='APPLIED COGNITIVE PSYCHOLOGY\nAppl. Cognit. Psychol. 20: 139–156 (2006)\nPublished online 31 October 2005 in Wiley InterScience\n(www.interscience.wiley.com) DOI: 10.1002/acp.1178\nConsequences of Erudite Vernacular Utilized Irrespective\nof Necessity: Problems with Using Long Words Needlessly\nDANIEL M. OPPENHEIMER*\nPrinceton University, USA\nSUMMARY\nMost texts on writing style encourage authors to avoid overly-complex words. However, a majority'), Document(metadata={'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'creator': 'Preview'

### Create embeddings


#### Understanding Embeddings: A Beginner's Guide

##### What are Embeddings?

Embeddings are numerical representations of text converted into vectors (lists of numbers). Think of them as a way to translate words, sentences, or documents into a language that machine learning models can understand and process.

**Example:**

Notice how "cat" and "dog" embeddings are very similar because they represent similar concepts.

---

#### Why Do We Need Embeddings?

1. **Semantic Understanding**: Embeddings capture the meaning of words. Similar words have similar embeddings.
2. **Machine Learning Ready**: Neural networks work with numbers, not raw text. Embeddings convert text to numbers.
3. **Similarity Search**: Find related documents quickly using vector distance (e.g., which documents are similar to a query?).
4. **Memory Efficient**: Instead of storing entire documents, store compressed numerical representations.
5. **RAG (Retrieval-Augmented Generation)**: Essential for retrieving relevant context from large documents to feed into LLMs.

---

#### How Are Embeddings Created?

##### The Process:

1. **Input Text**: You provide text (word, sentence, or document)
2. **Neural Network**: A pre-trained deep learning model processes the text
3. **Output Vector**: The model outputs a vector of numbers (e.g., 384 dimensions)
4. **Storage**: Store these vectors in a vector database for fast retrieval


In [4]:
embeddings =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_24680\632999143.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings =HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1006.61it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Store in vector DB


In [5]:
db = Chroma.from_documents(doc, embeddings)



In [6]:
#create retriever and get relevant documents
retriever = db.as_retriever()
relevant_chunks = retriever.invoke("what is the title of the paper?")
print(relevant_chunks)

[Document(metadata={'author': 'Thu Vu', 'page_label': '1', 'creationdate': "D:20240909152042Z00'00'", 'total_pages': 3, 'page': 0, 'source': 'data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf', 'producer': 'macOS Version 14.4.1 (Build 23E224) Quartz PDFContext, AppendMode 1.1', 'moddate': "D:20240910141854Z00'00'", 'creator': 'Preview', 'title': 'Oppenheimer-2006-Applied_Cognitive_Psychology'}, page_content='direction. Implications and applications are discussed. Copyright #2005 John Wiley & Sons, Ltd.\nWhen it comes to writing, most experts agree that clarity, simplicity and parsimony are\nideals that authors should strive for. In their classic manual of style, Strunk and White\n(1979) encourage authors to ‘omit needless words.’ Daryl Bem’s (1995) guidelines for\nsubmission to Psychological Bulletin advise, ‘the ﬁrst step towards clarity is writing'), Document(metadata={'author': 'Thu Vu', 'page_label': '2', 'creator': 'Preview', 'page': 1, 'title': 'Oppenheimer-2006-Applied_Cogn

In [7]:
#Prompt template
PROMPT_TEMPLATE = """
    You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer
    the question. If you don't know the answer, say that you
    don't know. DON'T MAKE UP ANYTHING.
{context}
---
Answer the question based on the above context: {question}
"""

In [8]:
# Concatenate context text
context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, 
                                question="What is the title of the paper?")
print(prompt)

Human: 
    You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer
    the question. If you don't know the answer, say that you
    don't know. DON'T MAKE UP ANYTHING.
direction. Implications and applications are discussed. Copyright #2005 John Wiley & Sons, Ltd.
When it comes to writing, most experts agree that clarity, simplicity and parsimony are
ideals that authors should strive for. In their classic manual of style, Strunk and White
(1979) encourage authors to ‘omit needless words.’ Daryl Bem’s (1995) guidelines for
submission to Psychological Bulletin advise, ‘the ﬁrst step towards clarity is writing

---

text’s authors.
EXPERIMENT 1
Experiment 1 aimed to answer several simple questions. First, does increasing the
complexity of text succeed in making the author appear more intelligent? Second, to
what extent does the success of this strategy depend on the quality of the original, simpler
writing? Finally, if the strategy is

### use local LLM


In [9]:
llm = Ollama(model="gemma3:4b")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_24680\1559220570.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma3:4b")


In [10]:
llm.invoke(prompt)

'The context does not provide the title of the paper. It only includes snippets from different publications discussing writing style and cognitive psychology.'

In [11]:
#Create RAG Chain
qa = RetrievalQA.from_chain_type(llm=llm, retriever=db.as_retriever())

In [13]:
# ask a question
answer = qa.invoke("what is the title of the paper?")
print(answer)

{'query': 'what is the title of the paper?', 'result': 'Based on the provided text, the title of the paper is “Problems with long words”.'}
